# <font color="#418FDE" size="6.5" uppercase>**Mixtures Covariance**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Fit Gaussian and Bayesian Gaussian mixture models for soft clustering and density modeling. 
- Select mixture complexity using covariance structures, AIC, and BIC. 
- Apply covariance and precision estimators for robust or structured covariance analysis. 


## **1. Gaussian Mixtures**

### **1.1. GMM Fitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_A/image_01_01.jpg?v=1784030135" width="250">



>* Models data as overlapping Gaussian groups
>* Assigns probabilities instead of hard labels

>* Estimate component size, center, and spread
>* Learn clusters and density without labels

>* Choose component counts carefully
>* Balance statistical fit with interpretation



In [ ]:
#@title Python Code - GMM Fitting

# Fit a Gaussian mixture to synthetic clusters.
# Show soft membership probabilities for overlapping points.
# Visualize learned components and uncertain boundaries.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture

# Create overlapping two-dimensional data for soft clustering.
centers = np.array([[-2.0, 0.0], [1.8, 0.4], [0.0, 2.8]])
X, true_labels = make_blobs(
    n_samples=450, centers=centers, cluster_std=1.05, random_state=42
)

# Validate the expected two-feature shape before modeling.
if X.shape != (450, 2):
    raise ValueError("Expected 450 rows and 2 numeric features.")

# Fit one Gaussian mixture with three bell-shaped components.
gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=42)
gmm.fit(X)

# Predict both hard labels and soft membership probabilities.
predicted_labels = gmm.predict(X)
probabilities = gmm.predict_proba(X)

# Find one point whose membership is most uncertain.
max_probabilities = probabilities.max(axis=1)
uncertain_index = np.argmin(max_probabilities)
uncertain_point = X[uncertain_index]
uncertain_probs = probabilities[uncertain_index]

print("scikit-learn version:", sklearn.__version__)
print("Component weights:", np.round(gmm.weights_, 2))
print("Most uncertain point:", np.round(uncertain_point, 2))
print("Its membership probabilities:", np.round(uncertain_probs, 2))

# Plot points colored by their most likely mixture component.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    X[:, 0], X[:, 1], c=predicted_labels, cmap="viridis", s=24, alpha=0.65
)

# Mark learned component centers and the uncertain example.
ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c="red", s=120, marker="X")
ax.scatter(uncertain_point[0], uncertain_point[1], c="black", s=120, marker="*")
ax.set_title("Gaussian Mixture Model Fitting")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(handles=scatter.legend_elements()[0], labels=["0", "1", "2"])
plt.show()



### **1.2. Covariance Structures**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_A/image_01_02.jpg?v=1784030137" width="250">



>* Covariance defines cluster shape and orientation
>* Shape affects soft membership probabilities

>* Flexible covariances capture complex cluster shapes
>* Simpler covariances improve stability and reduce overfitting

>* Covariance shapes component probability spread
>* Priors stabilize Bayesian mixture density estimates



In [ ]:
#@title Python Code - Covariance Structures

# This example compares Gaussian mixture covariance choices.
# Covariance controls each component shape and orientation.
# The plot shows soft clustering boundaries changing.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture

# Create two elongated clusters with a fixed random seed.
points, true_labels = make_blobs(
    n_samples=500,
    centers=2,
    cluster_std=1.0,
    random_state=42,
)

# Stretch and rotate the blobs to create correlated features.
transform = np.array([[2.4, 1.2], [-0.7, 0.8]])
points = points @ transform

# Validate the small two-feature dataset before modeling.
if points.shape != (500, 2):
    raise ValueError("Expected 500 rows and 2 features.")

# Fit one flexible full-covariance Gaussian mixture model.
full_model = GaussianMixture(
    n_components=2,
    covariance_type="full",
    random_state=42,
)
full_model.fit(points)

# Fit one restricted spherical-covariance Gaussian mixture model.
spherical_model = GaussianMixture(
    n_components=2,
    covariance_type="spherical",
    random_state=42,
)
spherical_model.fit(points)

# Compare model fit using lower BIC as the better balance.
full_bic = full_model.bic(points)
spherical_bic = spherical_model.bic(points)
print(f"scikit-learn version: {sklearn_version}")
print(f"Full covariance BIC: {full_bic:.1f}")
print(f"Spherical covariance BIC: {spherical_bic:.1f}")

# Predict soft membership probabilities for the flexible model.
probabilities = full_model.predict_proba(points)
confidence = probabilities.max(axis=1)
print(f"Average full-model membership confidence: {confidence.mean():.2f}")

# Build a grid to visualize the learned soft boundary.
x_min = points[:, 0].min() - 2.0
x_max = points[:, 0].max() + 2.0
y_min = points[:, 1].min() - 2.0
y_max = points[:, 1].max() + 2.0

# Evaluate component probability across the feature space.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 180),
    np.linspace(y_min, y_max, 180),
)
grid = np.c_[xx.ravel(), yy.ravel()]
soft_grid = full_model.predict_proba(grid)[:, 0].reshape(xx.shape)

# Plot data and the full-covariance soft decision boundary.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    points[:, 0],
    points[:, 1],
    c=confidence,
    cmap="viridis",
    s=22,
)

# Add the probability contour where both components are equally likely.
ax.contour(xx, yy, soft_grid, levels=[0.5], colors="black", linewidths=2)
ax.set_title("Full covariance GMM: soft clustering boundary")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")

# Add a colorbar to show membership confidence values.
colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label("Highest component probability")
plt.show()



### **1.3. EM Intuition**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_A/image_01_03.jpg?v=1784030139" width="250">



>* EM handles uncertain cluster membership softly
>* Responsibilities reveal blended Gaussian data patterns

>* Estimate soft memberships from current components
>* Update parameters until assignments stabilize

>* Replace hidden labels with weighted responsibilities
>* Use multiple starts to compare solutions



## **2. Mixture Model Selection**

### **2.1. AIC BIC**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_A/image_02_01.jpg?v=1784030141" width="250">



>* Balance mixture fit against model complexity
>* Compare components and covariance assumptions

>* AIC favors predictive detail with more complexity
>* BIC favors simpler, stable explanations

>* Compare component counts and covariance shapes
>* Balance better fit against overfitting risk



In [ ]:
#@title Python Code - AIC BIC

# Compare mixture models using AIC and BIC.
# Covariance choices change model complexity penalties.
# Lower criteria identify preferred candidate models.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture

# Create small overlapping clusters for mixture modeling.
centers = [(-3, -1), (0, 2), (3, -1)]
X, true_labels = make_blobs(
    n_samples=450, centers=centers, cluster_std=0.9, random_state=42
)

# Stretch the data so full covariance can help.
stretch = np.array([[1.0, 0.8], [0.2, 1.4]])
X = X @ stretch

# Validate the generated data before fitting models.
if X.shape != (450, 2):
    raise ValueError("Expected 450 rows and 2 features.")

# Fit candidate Gaussian mixtures with different complexity.
rows = []
for covariance_type in ["spherical", "diag", "full"]:
    for n_components in [1, 2, 3, 4, 5]:
        model = GaussianMixture(
            n_components=n_components,
            covariance_type=covariance_type,
            random_state=42,
            n_init=3,
        )
        model.fit(X)
        rows.append(
            [covariance_type, n_components, model.aic(X), model.bic(X)]
        )

# Summarize the best few candidates by BIC.
results = pd.DataFrame(rows, columns=["covariance", "components", "AIC", "BIC"])
results["AIC"] = results["AIC"].round(1)
results["BIC"] = results["BIC"].round(1)
best_aic = results.loc[results["AIC"].idxmin()]
best_bic = results.loc[results["BIC"].idxmin()]

print("scikit-learn version:", sklearn_version)
print("Best by AIC:", best_aic["covariance"], int(best_aic["components"]))
print("Best by BIC:", best_bic["covariance"], int(best_bic["components"]))
print(results.sort_values("BIC").head(5).to_string(index=False))

# Plot BIC curves to show the complexity trade-off.
fig, ax = plt.subplots(figsize=(7, 4))
for covariance_type in ["spherical", "diag", "full"]:
    subset = results[results["covariance"] == covariance_type]
    ax.plot(subset["components"], subset["BIC"], marker="o", label=covariance_type)

ax.set_title("BIC for Gaussian Mixture Candidates")
ax.set_xlabel("Number of mixture components")
ax.set_ylabel("BIC, lower is better")
ax.legend(title="Covariance")
plt.show()



### **2.2. Bayesian Mixtures**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_A/image_02_02.jpg?v=1784030142" width="250">



>* Use priors to judge mixture complexity
>* Represent soft cluster uncertainty without extra segments

>* Priors regularize mixtures to reduce overfitting
>* Simpler components need strong supporting evidence

>* Covariance choices shape needed mixture components
>* Bayesian mixtures support stable, interpretable selection



In [ ]:
#@title Python Code - Bayesian Mixtures

# Compare mixture complexity with Bayesian regularization.
# Covariance choices affect selected component counts.
# Printed scores reveal simpler supported structure.

import numpy as np
import sklearn
from sklearn.datasets import make_blobs
from sklearn.mixture import BayesianGaussianMixture
from sklearn.mixture import GaussianMixture

# Create overlapping clusters for a small density example.
features, true_labels = make_blobs(
    n_samples=450,
    centers=3,
    cluster_std=[0.7, 1.0, 1.3],
    random_state=42,
)

# Validate the generated data before modeling.
if features.shape != (450, 2):
    raise ValueError("Expected 450 rows and 2 features.")

if not np.isfinite(features).all():
    raise ValueError("All feature values must be finite.")

# Fit ordinary mixtures and compare AIC and BIC.
component_options = [1, 2, 3, 4, 5, 6]
best_bic = None
best_components = None

for component_count in component_options:
    model = GaussianMixture(
        n_components=component_count,
        covariance_type="full",
        random_state=42,
    )
    model.fit(features)
    bic_value = model.bic(features)
    if best_bic is None or bic_value < best_bic:
        best_bic = bic_value
        best_components = component_count

# Fit a Bayesian mixture with extra possible components.
bayesian_model = BayesianGaussianMixture(
    n_components=6,
    covariance_type="full",
    weight_concentration_prior_type="dirichlet_process",
    weight_concentration_prior=0.05,
    max_iter=500,
    random_state=42,
)

bayesian_model.fit(features)
active_weights = bayesian_model.weights_ > 0.05
active_components = int(np.sum(active_weights))

# Show how Bayesian weights shrink unsupported components.
rounded_weights = np.round(bayesian_model.weights_, 3)
print("scikit-learn version:", sklearn.__version__)
print("Best BIC component count:", best_components)
print("Bayesian active components above 0.05 weight:", active_components)
print("Bayesian component weights:", rounded_weights)
print("Lesson: extra allowed components can receive near-zero weight.")



### **2.3. Dirichlet Priors**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_A/image_02_03.jpg?v=1784030144" width="250">



>* Priors express uncertainty about component weights
>* Prior strength influences active mixture components

>* Concentration controls active mixture components
>* Sparse priors reduce overfitting noise

>* AIC and BIC penalize extra components
>* Dirichlet priors guide flexible component selection



In [ ]:
#@title Python Code - Dirichlet Priors

# This example compares Dirichlet prior strengths.
# Sparse priors can deactivate extra mixture components.
# The plot shows learned component weights.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.mixture import BayesianGaussianMixture
import sklearn

# Generate three clear clusters for mixture modeling.
features, true_labels = make_blobs(
    n_samples=450,
    centers=3,
    cluster_std=0.65,
    random_state=42,
)

# Validate the generated data shape before modeling.
if features.shape != (450, 2):
    raise ValueError("Expected 450 rows and 2 features.")

# Fit two models that start with too many components.
prior_values = [0.01, 10.0]
learned_weights = []

for prior in prior_values:
    model = BayesianGaussianMixture(
        n_components=8,
        weight_concentration_prior=prior,
        random_state=42,
        max_iter=500,
    )
    model.fit(features)
    learned_weights.append(model.weights_)

# Count components with meaningful learned weight.
active_counts = []
for weights in learned_weights:
    active_counts.append(int(np.sum(weights > 0.05)))

print("scikit-learn version:", sklearn.__version__)
print("Sparse prior active components:", active_counts[0])
print("Strong prior active components:", active_counts[1])

# Plot learned weights for each possible component.
component_numbers = np.arange(1, 9)
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(component_numbers, learned_weights[0], marker="o", label="prior = 0.01")
ax.plot(component_numbers, learned_weights[1], marker="s", label="prior = 10.0")
ax.set_title("Dirichlet Prior Strength Changes Mixture Complexity")
ax.set_xlabel("Component number")
ax.set_ylabel("Learned mixture weight")
ax.legend()
plt.show()



## **3. Covariance Models**

### **3.1. Empirical Shrinkage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_A/image_03_01.jpg?v=1784030130" width="250">



>* Raw covariance can be noisy and unstable
>* Shrinkage pulls estimates toward stable structure

>* Shrinkage stabilizes mixture covariance estimates
>* Prevents noisy, overly narrow cluster shapes

>* Data-driven shrinkage adapts to sample uncertainty
>* Balances flexible structure with reliable covariance patterns



In [ ]:
#@title Python Code - Empirical Shrinkage

# This example compares empirical and shrinkage covariance estimates.
# Shrinkage stabilizes covariance when samples are limited.
# The plot shows improved conditioning after shrinkage.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.covariance import EmpiricalCovariance
from sklearn.covariance import LedoitWolf

# A fixed generator makes the simulated data reproducible.
rng = np.random.default_rng(42)

# We create more features than observations to stress covariance estimation.
n_samples = 30
n_features = 20

# Shared hidden factors create real correlations among features.
latent = rng.normal(size=(n_samples, 3))
loadings = rng.normal(size=(3, n_features))
noise = 0.7 * rng.normal(size=(n_samples, n_features))

# Centering keeps the covariance comparison focused on spread.
data = latent @ loadings + noise
data = data - data.mean(axis=0)

# Validate the intended high-dimensional teaching setup.
if data.shape != (n_samples, n_features):
    raise ValueError("Data shape should be samples by features.")

# Fit the raw empirical covariance and a shrinkage estimator.
empirical_model = EmpiricalCovariance().fit(data)
shrinkage_model = LedoitWolf().fit(data)

# Eigenvalues summarize stability of each covariance matrix.
empirical_eigenvalues = np.linalg.eigvalsh(empirical_model.covariance_)
shrinkage_eigenvalues = np.linalg.eigvalsh(shrinkage_model.covariance_)

# Condition number compares largest and smallest covariance directions.
empirical_condition = np.linalg.cond(empirical_model.covariance_)
shrinkage_condition = np.linalg.cond(shrinkage_model.covariance_)

print("scikit-learn version:", sklearn.__version__)
print("Ledoit-Wolf shrinkage amount:", round(shrinkage_model.shrinkage_, 3))
print("Empirical condition number:", round(empirical_condition, 1))
print("Shrinkage condition number:", round(shrinkage_condition, 1))

# Plot eigenvalues to show how shrinkage moderates extreme directions.
fig, ax = plt.subplots(figsize=(7, 4))
feature_numbers = np.arange(1, n_features + 1)

ax.plot(feature_numbers, empirical_eigenvalues, marker="o", label="Empirical")
ax.plot(feature_numbers, shrinkage_eigenvalues, marker="o", label="Shrinkage")
ax.set_title("Covariance Eigenvalues Before and After Shrinkage")
ax.set_xlabel("Eigenvalue rank")
ax.set_ylabel("Covariance eigenvalue")
ax.legend()
plt.show()



### **3.2. Sparse Precision**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_A/image_03_02.jpg?v=1784030132" width="250">



>* Precision matrices reveal direct conditional relationships
>* Sparsity filters indirect correlations in complex data

>* Regularization stabilizes high-dimensional covariance estimates
>* Graphs reveal direct links in complex data

>* Supports interpretable decisions using direct relationships
>* Balance sparsity with data quality and stability



In [ ]:
#@title Python Code - Sparse Precision

# This example estimates sparse precision relationships.
# GraphicalLasso reveals direct conditional associations.
# The plot compares covariance and precision patterns.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.covariance import GraphicalLasso
import sklearn

# A fixed generator makes the synthetic data reproducible.
rng = np.random.default_rng(42)

# Five variables are built from two hidden factors.
samples = 180
factor_a = rng.normal(size=samples)
factor_b = rng.normal(size=samples)
noise = rng.normal(scale=0.35, size=(samples, 5))

# Some variables share indirect correlation through factors.
data = np.column_stack(
    [factor_a, factor_a + noise[:, 1], factor_b, factor_b + noise[:, 3], noise[:, 4]]
)

# Validate the small teaching dataset before modeling.
if data.shape != (180, 5):
    raise ValueError("Expected 180 rows and 5 variables.")

# Centering helps covariance estimates focus on relationships.
centered_data = data - data.mean(axis=0)

# GraphicalLasso estimates a sparse inverse covariance matrix.
model = GraphicalLasso(alpha=0.18, max_iter=200)
model.fit(centered_data)

# Convert precision values to partial correlations for readability.
precision = model.precision_
diagonal = np.sqrt(np.diag(precision))
partial_corr = -precision / np.outer(diagonal, diagonal)
np.fill_diagonal(partial_corr, 1.0)

# Count nonzero off-diagonal conditional links.
link_mask = np.abs(partial_corr) > 0.05
nonzero_links = int((np.sum(link_mask) - 5) / 2)

print("scikit-learn version:", sklearn.__version__)
print("Variables:", data.shape[1], "Samples:", data.shape[0])
print("Estimated direct links:", nonzero_links)

# Show the sparse conditional relationship pattern.
fig, ax = plt.subplots(figsize=(5.5, 4.5))
image = ax.imshow(partial_corr, vmin=-1, vmax=1, cmap="coolwarm")

ax.set_title("Sparse precision as partial correlations")
ax.set_xlabel("Variable index")
ax.set_ylabel("Variable index")
fig.colorbar(image, ax=ax, label="Partial correlation")
plt.show()



### **3.3. Robust Covariance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_26/Lecture_A/image_03_03.jpg?v=1784030133" width="250">



>* Outliers can distort covariance estimates
>* Robust methods capture the main pattern

>* Estimate the main data cloud shape
>* Down-weight outliers for reliable distances

>* Robust precision reveals reliable conditional relationships
>* Limits outlier distortion in mixture modeling



In [ ]:
#@title Python Code - Robust Covariance

# This example compares ordinary and robust covariance.
# Outliers can distort covariance based distance estimates.
# Robust covariance better describes the main data cloud.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.covariance import EmpiricalCovariance
from sklearn.covariance import MinCovDet

# Generate one compact two dimensional cloud.
rng = np.random.default_rng(42)
main_cloud = rng.multivariate_normal([0, 0], [[1.0, 0.7], [0.7, 1.0]], 120)

# Add a few extreme observations that should not define normal spread.
outliers = np.array([[6.0, 6.5], [7.0, 5.5], [5.5, 7.0], [8.0, 6.0]])
data = np.vstack([main_cloud, outliers])

# Validate the small teaching dataset before fitting estimators.
if data.shape != (124, 2):
    raise ValueError("Expected 124 rows and 2 columns.")

# Fit classical and robust covariance estimators to the same data.
ordinary_covariance = EmpiricalCovariance().fit(data)
robust_covariance = MinCovDet(random_state=42).fit(data)

# Compare how far the outliers look under each covariance estimate.
ordinary_distances = ordinary_covariance.mahalanobis(data)
robust_distances = robust_covariance.mahalanobis(data)
ordinary_outlier_mean = ordinary_distances[-4:].mean()
robust_outlier_mean = robust_distances[-4:].mean()

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Ordinary covariance determinant: {np.linalg.det(ordinary_covariance.covariance_):.2f}")
print(f"Robust covariance determinant: {np.linalg.det(robust_covariance.covariance_):.2f}")
print(f"Mean outlier distance, ordinary: {ordinary_outlier_mean:.1f}")
print(f"Mean outlier distance, robust: {robust_outlier_mean:.1f}")

# Draw covariance ellipses from eigenvalues and eigenvectors.
angles = np.linspace(0, 2 * np.pi, 200)
circle = np.column_stack([np.cos(angles), np.sin(angles)])

# Convert each covariance matrix into a two standard deviation ellipse.
def covariance_ellipse(center, covariance):
    values, vectors = np.linalg.eigh(covariance)
    radii = 2.0 * np.sqrt(values)
    return center + circle @ np.diag(radii) @ vectors.T

ordinary_ellipse = covariance_ellipse(
    ordinary_covariance.location_, ordinary_covariance.covariance_
)
robust_ellipse = covariance_ellipse(
    robust_covariance.location_, robust_covariance.covariance_
)

# Plot the data and both estimated covariance shapes.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(main_cloud[:, 0], main_cloud[:, 1], s=25, label="main cloud")
ax.scatter(outliers[:, 0], outliers[:, 1], s=45, label="outliers")

ax.plot(ordinary_ellipse[:, 0], ordinary_ellipse[:, 1], label="ordinary covariance")
ax.plot(robust_ellipse[:, 0], robust_ellipse[:, 1], label="robust covariance")
ax.set_title("Robust covariance resists a few extreme points")
ax.set_xlabel("Feature 1")

ax.set_ylabel("Feature 2")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Mixtures Covariance**</font>


In this lecture, you learned to:
- Fit Gaussian and Bayesian Gaussian mixture models for soft clustering and density modeling. 
- Select mixture complexity using covariance structures, AIC, and BIC. 
- Apply covariance and precision estimators for robust or structured covariance analysis. 

In the next Lecture (Lecture B), we will go over 'Density Anomalies'